In [1]:
import pandas as pd

In [2]:
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")

In [3]:
train_data

,year,month,day,order,country,session_id,page1_main_category,page2_clothing_model,colour,location,model_photography,price,price_2,page
0,2008,6,22,21,29,15648,3,C20,13,1,2,48,1,2
1,2008,5,19,6,29,10018,2,B26,13,3,1,57,1,2
2,2008,7,15,2,29,19388,3,C13,9,5,1,48,1,1
3,2008,5,2,2,29,7181,2,B11,2,4,1,43,2,1
4,2008,6,9,16,29,13493,2,B31,9,5,1,57,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132374,2008,7,4,3,29,17622,4,P19,2,1,1,48,1,2
132375,2008,6,19,9,29,15165,3,C26,14,3,1,28,2,2
132376,2008,7,15,4,29,19359,1,A4,3,2,2,38,2,1
132377,2008,7,28,16,29,21454,3,C50,9,5,2,20,2,3


In [4]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132379 entries, 0 to 132378
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   year                  132379 non-null  int64 
 1   month                 132379 non-null  int64 
 2   day                   132379 non-null  int64 
 3   order                 132379 non-null  int64 
 4   country               132379 non-null  int64 
 5   session_id            132379 non-null  int64 
 6   page1_main_category   132379 non-null  int64 
 7   page2_clothing_model  132379 non-null  object
 8   colour                132379 non-null  int64 
 9   location              132379 non-null  int64 
 10  model_photography     132379 non-null  int64 
 11  price                 132379 non-null  int64 
 12  price_2               132379 non-null  int64 
 13  page                  132379 non-null  int64 
dtypes: int64(13), object(1)
memory usage: 14.1+ MB


In [5]:

from sklearn.preprocessing import LabelEncoder
le1 = LabelEncoder()
train_data['page2_clothing_model'] = le1.fit_transform(train_data['page2_clothing_model'])

le2 = LabelEncoder()
test_data['page2_clothing_model'] = le2.fit_transform(test_data['page2_clothing_model'])

In [6]:
from sklearn.preprocessing import StandardScaler
train_features = train_data[['page1_main_category', 'page2_clothing_model', 'colour']]
train_target = train_data['price']

test_features = test_data[['page1_main_category', 'page2_clothing_model', 'colour']]
test_target = test_data['price']

scaler = StandardScaler()
train_features = scaler.fit_transform(train_features)
test_features = scaler.transform(test_features)


In [7]:
test_features

array([[ 1.39700767,  1.58533737,  0.65417332],
       [-1.22307939, -1.30004534,  1.83388111],
       [ 1.39700767,  1.1270707 , -0.05365136],
       ...,
       [-0.34971704, -0.12891942, -1.23335914],
       [ 0.52364532,  0.34632008,  0.1822902 ],
       [ 0.52364532,  0.15961885,  1.36199799]])

In [8]:

train_features.shape

(132379, 3)

In [9]:

#Model Evaluation and Hyperparameter Tuning


In [20]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score


In [24]:
model_params = {
    "Linear Regressor": (LinearRegression(), {}),

    "Ridge Regressor": (Ridge(), {
        "alpha": [0.01, 0.1, 1, 10, 100]
    }),

    "Lasso Regressor": (Lasso(), {
        "alpha": [0.01, 0.1, 1, 10, 100]
    }),

    "Gradient Boosting Regressor": (GradientBoostingRegressor(), {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.01, 0.1, 0.2],
        "max_depth": [3, 5, 7]
    }),

    "Random Forest Regressor": (RandomForestRegressor(), {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5, 10]
    })
}

In [ ]:
for name, (model, param_grid) in model_params.items():

    print(f"Training {name}...")

    if param_grid:

        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            n_iter=3,
            cv=3,
            scoring="r2",
            n_jobs=1,
            random_state=42
        )

        random_search.fit(train_features, train_target)

        best_model = random_search.best_estimator_
        best_params = random_search.best_params_

    else:

        best_model = model
        best_model.fit(train_features, train_target)
        best_params = "Default Parameters"

    predictions = best_model.predict(test_features)

    mae = mean_absolute_error(test_target, predictions)
    r2 = r2_score(test_target, predictions)

    reports.append((name, best_model, best_params, mae, r2))



Training Linear Regressor...
Training Ridge Regressor...
Training Lasso Regressor...
Training Gradient Boosting Regressor...
Training Random Forest Regressor...

MODEL COMPARISON RESULTS



In [26]:
for name, model, best_params, mae, r2 in reports:
    print(f"Model: {name}")
    print(f"Best Parameters: {best_params}")
    print(f"MAE: {mae:.4f}")
    print(f"R2 Score: {r2:.4f}")
    print("\n")

best_model_name, best_model, best_params, best_mae, best_r2 = min(
    reports,
    key=lambda x: x[3]
)

print("BEST MODEL")
print(f"Model: {best_model_name}")
print(f"Best Parameters: {best_params}")
print(f"MAE: {best_mae:.4f}")
print(f"R2 Score: {best_r2:.4f}")

Model: Linear Regressor
Best Parameters: Default Parameters
MAE: 10.0456
R2 Score: 0.1364


Model: Ridge Regressor
Best Parameters: {'alpha': 0.1}
MAE: 10.0456
R2 Score: 0.1364


Model: Lasso Regressor
Best Parameters: {'alpha': 0.01}
MAE: 10.0402
R2 Score: 0.1364


Model: Gradient Boosting Regressor
Best Parameters: {'n_estimators': 100, 'max_depth': 5}
MAE: 2.0837
R2 Score: 0.9394


Model: Random Forest Regressor
Best Parameters: {'n_estimators': 100, 'max_depth': 10}
MAE: 1.1381
R2 Score: 0.9276


Model: Linear Regressor
Best Parameters: Default Parameters
MAE: 10.0456
R2 Score: 0.1364


Model: Ridge Regressor
Best Parameters: {'alpha': 0.1}
MAE: 10.0456
R2 Score: 0.1364


Model: Lasso Regressor
Best Parameters: {'alpha': 0.1}
MAE: 10.0135
R2 Score: 0.1355


Model: Gradient Boosting Regressor
Best Parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
MAE: 1.0514
R2 Score: 0.9607


Model: Random Forest Regressor
Best Parameters: {'n_estimators': 200, 'min_samples_sp

In [27]:
import pickle

with open('le1_clothing_model.pkl', 'wb') as f:
    pickle.dump(le1, f)

with open('le2_clothing_model.pkl', 'wb') as f:
    pickle.dump(le2, f)

with open('regression_standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('best_regression_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
